### Mini Project IX

#### Part 3 — Transformer Fine-Tuning & Error Analysis

Fine-tune DistilBERT on the moderation dataset, then evaluate and inspect misclassifications.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("✅ Imports loaded")

In [ ]:
# Configuration
NUM_LABELS = 3
EPOCHS     = 4
LR         = 3e-5
BATCH_SIZE = 32
MAX_LEN    = 128

CLASS_NAMES = {0: "Hate Speech", 1: "Offensive", 2: "Neither"}
LABELS = list(CLASS_NAMES.values())
PALETTE = {"Hate Speech": "#d62728", "Offensive": "#ff7f0e", "Neither": "#2ca02c"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load preprocessed splits from Part 1
X_train = np.load("X_train.npy", allow_pickle=True)
X_val   = np.load("X_val.npy",   allow_pickle=True)
X_test  = np.load("X_test.npy",  allow_pickle=True)

y_train = np.load("y_train.npy", allow_pickle=True)
y_val   = np.load("y_val.npy",   allow_pickle=True)
y_test  = np.load("y_test.npy",  allow_pickle=True)

# Dataset class
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx]
        }

# Tokenizer + datasets
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_dataset = TweetDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = TweetDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = TweetDataset(X_test,  y_test,  tokenizer, MAX_LEN)

# Weighted sampler to mitigate class imbalance
class_counts  = np.bincount(y_train)
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_train]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# Model setup
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_LABELS
)
model.to(device)

# Weighted loss for class imbalance
loss_weights = torch.tensor(class_weights / class_weights.sum(), dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=loss_weights * NUM_LABELS)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# Training helpers

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            input_ids  = batch["input_ids"].to(device)
            attn_mask  = batch["attention_mask"].to(device)
            labels     = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            loss    = criterion(outputs.logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item() * labels.size(0)
            preds       = outputs.logits.argmax(dim=-1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / total, correct / total


def get_predictions(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels    = batch["labels"].to(device)
            outputs   = model(input_ids=input_ids, attention_mask=attn_mask)
            preds     = outputs.logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# Train
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    vl_loss, vl_acc = run_epoch(val_loader,   training=False)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"Val loss {vl_loss:.4f} acc {vl_acc:.4f}")

# Save model weights
torch.save(model.state_dict(), "distilbert_safespace.pt")
print("Model saved.")

# Plot training curves
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("DistilBERT Training Curves", fontsize=13, fontweight="bold")

axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train")
axes[0].plot(epochs_range, history["val_loss"],   marker="o", label="Val")
axes[0].set_title("Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], marker="o", label="Train")
axes[1].plot(epochs_range, history["val_acc"],   marker="o", label="Val")
axes[1].set_title("Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()
plt.savefig("transformer_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Evaluate on test set

y_true, y_pred = get_predictions(test_loader)

print(f"\nTest accuracy : {accuracy_score(y_true, y_pred):.4f}")
print("\nTest classification report:")
print(classification_report(y_true, y_pred, target_names=LABELS))

# Save metrics for later comparison
report_dict = classification_report(
    y_true, y_pred, target_names=LABELS, output_dict=True
)
pd.DataFrame(report_dict).T.to_csv("transformer_metrics.csv")

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("DistilBERT — Test Set Confusion Matrix", fontsize=13, fontweight="bold")

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("Counts")
axes[0].set_xticklabels(LABELS, rotation=15, ha="right")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS,
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Row-normalised (Recall)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
axes[1].set_xticklabels(LABELS, rotation=15, ha="right")

plt.tight_layout()
plt.savefig("transformer_confusion.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Comparative analysis & error analysis

# Load baseline metrics (from Part 2)
baseline_metrics = pd.read_csv("baseline_metrics.csv", index_col=0)
transformer_metrics = pd.read_csv("transformer_metrics.csv", index_col=0)

# Obtain prediction probabilities for error analysis
import torch.nn.functional as F

model.eval()
all_probs, all_preds, all_true = [], [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels    = batch["labels"]
        logits    = model(input_ids=input_ids, attention_mask=attn_mask).logits
        probs     = F.softmax(logits, dim=-1).cpu()
        all_probs.append(probs)
        all_preds.extend(probs.argmax(dim=-1).numpy())
        all_true.extend(labels.numpy())

all_probs = torch.cat(all_probs, dim=0).numpy()
all_preds = np.array(all_preds)
all_true  = np.array(all_true)
confidence = all_probs.max(axis=1)

# 1) Side-by-side model comparison
metrics_rows = ["Hate Speech", "Offensive", "Neither", "macro avg", "weighted avg"]
compare_cols = ["precision", "recall", "f1-score"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Baseline vs DistilBERT — Test Set Metrics", fontsize=13, fontweight="bold")

for ax, metric in zip(axes, compare_cols):
    base_vals  = baseline_metrics.loc[metrics_rows, metric].values.astype(float)
    trans_vals = transformer_metrics.loc[metrics_rows, metric].values.astype(float)
    x = np.arange(len(metrics_rows))
    w = 0.35
    ax.bar(x - w/2, base_vals,  w, label="TF-IDF + LR",  color="#5b9bd5", alpha=0.85)
    ax.bar(x + w/2, trans_vals, w, label="DistilBERT",    color="#ed7d31", alpha=0.85)
    ax.set_title(metric.capitalize())
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_rows, rotation=15, ha="right", fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color="grey", linewidth=0.7, linestyle="--")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Metric delta (DistilBERT − Baseline):")
delta = transformer_metrics.loc[metrics_rows, compare_cols].astype(float) - \
        baseline_metrics.loc[metrics_rows, compare_cols].astype(float)
print(delta.round(4).to_string())

# 2) Per-class trouble spots
print("\nPer-class F1 — Baseline:")
print(baseline_metrics.loc[LABELS, "f1-score"].round(4))
print("\nPer-class F1 — DistilBERT:")
print(transformer_metrics.loc[LABELS, "f1-score"].round(4))

# 3) Error analysis for transformer
errors_mask = all_preds != all_true
error_texts  = X_test[errors_mask]
error_true   = all_true[errors_mask]
error_pred   = all_preds[errors_mask]
error_conf   = confidence[errors_mask]

np.random.seed(SEED)
error_df = pd.DataFrame({
    "tweet":      error_texts,
    "true_label": [CLASS_NAMES[c] for c in error_true],
    "pred_label": [CLASS_NAMES[c] for c in error_pred],
    "confidence": error_conf.round(3)
})

sampled = (error_df.groupby("true_label", group_keys=False)
                   .apply(lambda g: g.sample(min(len(g), 4), random_state=SEED)))
sampled = sampled.sample(frac=1, random_state=SEED).head(12).reset_index(drop=True)

# Tag failure categories

def categorise_error(row):
    text = row["tweet"].lower()
    if any(w in text for w in ["lol", "haha", "jk", "/s", "irony"]):
        return "Sarcasm / Irony"
    if any(w in text for w in ["[user]", "[url]"]):
        return "Context-Dependent"
    if len(text.split()) < 6:
        return "Too Short / Ambiguous"
    if row["true_label"] == "Hate Speech" and row["pred_label"] == "Offensive":
        return "Hate↔Offensive Confusion"
    if row["confidence"] < 0.60:
        return "Annotation Disagreement"
    return "Slang / Informal Language"

sampled["failure_category"] = sampled.apply(categorise_error, axis=1)

print("\nMisclassified examples (transformer):")
pd.set_option("display.max_colwidth", 80)
print(sampled[["tweet", "true_label", "pred_label", "confidence", "failure_category"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
cat_counts = sampled["failure_category"].value_counts()
ax.barh(cat_counts.index, cat_counts.values, color="#ed7d31", alpha=0.85)
ax.set_title("Transformer Error Categories", fontweight="bold")
ax.set_xlabel("Count")
for i, v in enumerate(cat_counts.values):
    ax.text(v + 0.05, i, str(v), va="center")
plt.tight_layout()
plt.savefig("error_categories.png", dpi=150, bbox_inches="tight")
plt.show()

# 4) Confidence threshold analysis
thresholds = np.linspace(0.3, 0.99, 60)
coverages, accuracies = [], []

for t in thresholds:
    mask = confidence >= t
    if mask.sum() == 0:
        break
    coverages.append(mask.mean())
    accuracies.append(accuracy_score(all_true[mask], all_preds[mask]))

fig, ax1 = plt.subplots(figsize=(9, 5))
fig.suptitle("Accuracy vs Coverage at Confidence Thresholds", fontweight="bold")

color_acc, color_cov = "#ed7d31", "#5b9bd5"
ax1.plot(thresholds[:len(accuracies)], accuracies, color=color_acc,
         linewidth=2, label="Accuracy")
ax1.set_xlabel("Confidence Threshold")
ax1.set_ylabel("Accuracy", color=color_acc)
ax1.tick_params(axis="y", labelcolor=color_acc)
ax1.set_ylim(0.5, 1.0)

ax2 = ax1.twinx()
ax2.plot(thresholds[:len(coverages)], coverages, color=color_cov,
         linewidth=2, linestyle="--", label="Coverage")
ax2.set_ylabel("Coverage (fraction predicted)", color=color_cov)
ax2.tick_params(axis="y", labelcolor=color_cov)
ax2.set_ylim(0, 1.05)

op_thresh = 0.75
op_idx    = np.argmin(np.abs(thresholds[:len(accuracies)] - op_thresh))
ax1.axvline(op_thresh, color="grey", linestyle=":", linewidth=1.5)
ax1.annotate(
    f"t={op_thresh}\nacc={accuracies[op_idx]:.2f}\ncov={coverages[op_idx]:.2f}",
    xy=(op_thresh, accuracies[op_idx]),
    xytext=(op_thresh + 0.04, accuracies[op_idx] - 0.08),
    fontsize=8,
    arrowprops=dict(arrowstyle="->", color="grey"),
)

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc="lower right")

plt.tight_layout()
plt.savefig("confidence_coverage.png", dpi=150, bbox_inches="tight")
plt.show()
